# 8주차 과제 문제와 정답
학생에게 공개하지 않는 교수용 풀이입니다. 문제 본문은 `week08_assignment.qmd`에 있으며, 각 풀이는 출력 예시와 같은 결과를 내도록 검증했습니다.

**채점 참고**: 다른 입력으로도 시험해 일반적으로 동작하는지 확인합니다.

## 과제 1. 캠핑 모임 경비 정산
**하위 문제**: ① 인원 확인 ② 나누어 낼 총액 ③ 100원 단위 올림 ④ 걷는 총액과 남는 금액 출력

**채점 참고 입력**: `900000 / 30 / 0` → 30,000원 / 900,000원 / 0원 (딱 나누어떨어지는 경우), `1250000 / 28 / 300000` → 34,000원

In [ ]:
import math

def settle(total_cost, people, subsidy):
    if people < 1:
        return None
    burden_total = max(total_cost - subsidy, 0)
    per_person = math.ceil(burden_total / people / 100) * 100
    collected = per_person * people
    return per_person, collected, collected - burden_total

# 학생 제출 형태 (input 사용)
# total_cost = int(input())
# people = int(input())
# subsidy = int(input())
result = settle(1250000, 28, 300000)
if result is None:
    print("인원은 1명 이상이어야 합니다.")
else:
    per_person, collected, left = result
    print(f"1인 부담금: {per_person:,}원")
    print(f"걷는 총액: {collected:,}원")
    print(f"남는 금액: {left:,}원")

## 과제 2. 제품 검수 점수 분석
**하위 문제**: ① 등급 함수 ② 합계·최고·최저·재검사·등급별 집계 ③ 정렬해 상위 3명

**채점 참고**: 목록을 바꿔도 동작해야 합니다(동점 없음 가정). 59점·60점·90점이 경계값입니다.

In [ ]:
products = [
    ("무선 이어폰", 78), ("보조 배터리", 92), ("전기 주전자", 45), ("스피커", 88), ("스마트 워치", 60),
    ("로봇 청소기", 100), ("공기 청정기", 59), ("전동 칫솔", 73), ("선풍기", 85), ("무선 충전기", 90),
]

def get_grade(score):
    if score >= 90:
        return "A"
    if score >= 80:
        return "B"
    if score >= 70:
        return "C"
    if score >= 60:
        return "D"
    return "F"

def analyze(products):
    total = 0
    best = products[0]
    worst = products[0]
    counts = {"A": 0, "B": 0, "C": 0, "D": 0, "F": 0}
    retest = []

    for name, score in products:
        total += score
        if score > best[1]:
            best = (name, score)
        if score < worst[1]:
            worst = (name, score)
        counts[get_grade(score)] += 1
        if score < 60:
            retest.append((name, score))

    pairs = []
    for name, score in products:
        pairs.append((score, name))
    pairs.sort(reverse=True)

    print(f"평균: {total / len(products):.2f}점")
    print(f"최고점: {best[0]} ({best[1]}점)")
    print(f"최저점: {worst[0]} ({worst[1]}점)")
    print(f"재검사 대상 ({len(retest)}개)")
    for name, score in retest:
        print(f"- {name} {score}점")
    for grade in counts:
        print(f"{grade}등급 {counts[grade]}개")
    for place, pair in enumerate(pairs[:3], start=1):
        score, name = pair
        print(f"{place}위 {name} {score}점")

analyze(products)

## 과제 3. 글자 빈도 세기
**하위 문제**: ① 공백 제외하고 세기(`continue`) ② 빈 입력 처리 ③ `(-횟수, 글자)` 튜플 정렬 ④ 최대 3개 출력

**채점 참고 입력**: `가나다 가나` → 글자 수 5, 서로 다른 글자 3, 1위 가 2회 / 2위 나 2회 / 3위 다 1회

In [ ]:
def char_freq(text):
    counts = {}
    total = 0
    for ch in text:
        if ch == " ":
            continue
        total += 1
        counts[ch] = counts.get(ch, 0) + 1

    if total == 0:
        print("입력된 글자가 없습니다.")
        return

    pairs = []
    for ch, n in counts.items():
        pairs.append((-n, ch))      # 횟수는 큰 순서, 글자는 가나다순이 되도록 정렬
    pairs.sort()

    print(f"글자 수(공백 제외): {total}")
    print(f"서로 다른 글자: {len(counts)}")
    print("많이 나온 글자")
    for place, pair in enumerate(pairs[:3], start=1):
        neg, ch = pair
        print(f"{place}위 {ch} {-neg}회")

# 학생 제출 형태: text = input()
char_freq("푸른 하늘 푸른 바다")

## 과제 4. 회의실 예약 충돌 검사
**하위 문제**: ① 잘못된 기록 분리 ② 겹침 판단 함수(`a_시작 < b_종료 and b_시작 < a_종료`) ③ 둘씩 짝짓기(중첩 반복, `j`는 `i+1`부터) ④ 집계

**채점 참고**: 끝 시각 = 시작 시각이면 겹치지 않음(9~10시와 10~12시). 모든 예약이 겹치는 자료, 하나도 겹치지 않는 자료, 잘못된 기록만 있는 자료도 시험합니다.

In [ ]:
bookings = [
    ("기획 회의", 10, 12), ("디자인 검토", 11, 13), ("고객 상담", 13, 15),
    ("팀 워크숍", 14, 16), ("아침 보고", 9, 10), ("임시 예약", 16, 16),
]

def overlaps(a_start, a_end, b_start, b_end):
    return a_start < b_end and b_start < a_end

def check(bookings):
    valid = []
    invalid = []
    for name, start, end in bookings:
        if start < end:
            valid.append((name, start, end))
        else:
            invalid.append((name, start, end))

    pairs = []
    in_conflict = []
    for i in range(len(valid)):
        for j in range(i + 1, len(valid)):
            a = valid[i]
            b = valid[j]
            if overlaps(a[1], a[2], b[1], b[2]):
                pairs.append((a[0], b[0]))
                in_conflict.append(a[0])
                in_conflict.append(b[0])

    free = 0
    hours = 0
    for name, start, end in valid:
        hours += end - start
        if name not in in_conflict:
            free += 1

    print(f"잘못된 기록 ({len(invalid)}건)")
    for name, start, end in invalid:
        print(f"- {name} {start}시~{end}시")
    print(f"겹치는 예약 ({len(pairs)}쌍)")
    for a, b in pairs:
        print(f"- {a} / {b}")
    print(f"겹치지 않는 예약 수: {free}")
    print(f"총 사용 시간: {hours}시간")

check(bookings)

## 과제 5. 대기줄 시뮬레이션
**하위 문제**: ① 창구가 비는 시각을 리스트로 기록 ② 가장 먼저 비는 창구 고르기(동률은 번호가 작은 쪽) ③ 시작 시각 = `max(도착, 창구가 빌 때)` ④ 집계

**채점 참고 자료**: `arrivals = [0,0,0,1,1,2]`, `services = [4,4,4,2,2,1]`
- 창구 1개: 평균 대기 8.33분, 최대 14분, 대기한 손님 5명, 마지막 종료 17분
- 창구 2개: 평균 대기 3.00분, 최대 6분, 대기한 손님 4명, 마지막 종료 9분

In [ ]:
arrivals = [0, 2, 3, 7, 8, 16]
services = [3, 4, 2, 5, 1, 3]

def simulate(arrivals, services, counters):
    free_at = [0] * counters       # 각 창구가 비는 시각
    waits = []
    last_end = 0

    for n in range(len(arrivals)):
        k = 0
        for i in range(counters):
            if free_at[i] < free_at[k]:
                k = i
        start = max(arrivals[n], free_at[k])
        waits.append(start - arrivals[n])
        free_at[k] = start + services[n]
        if free_at[k] > last_end:
            last_end = free_at[k]

    waited = 0
    for w in waits:
        if w > 0:
            waited += 1
    return sum(waits) / len(waits), max(waits), waited, last_end

for counters in (1, 2, 3):
    avg, longest, waited, end = simulate(arrivals, services, counters)
    print(f"창구 {counters}개: 평균 대기 {avg:.2f}분, 최대 대기 {longest}분, 대기한 손님 {waited}명, 마지막 종료 {end}분")